In [ ]:
import os
from openai import OpenAI
import time

# Инициализация клиента
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key="HF_KEY",
)

# Список URL изображений 
image_urls = [
'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple1.jpeg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple2.jpeg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple3.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple4.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple5.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple6.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple7.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple8.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple9.jpg',
  'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate1.jpeg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate2.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate3.png',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate4.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate5.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate6.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate7.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate8.jpg',
    'https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/granate9.jpg'
]

# Истинные метки 
true_labels = ['Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 'Apple', 
'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate', 'Pomegranate']  

# Модели для сравнения
models = {
    'Qwen': 'Qwen/Qwen3-VL-8B-Instruct:novita',
    'Gemma': 'google/gemma-3-27b-it:nebius' 
}

In [43]:
def get_prediction(model_id, image_url):
    """Получить предсказание от модели для одного изображения"""
    try:
        completion = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Identify the fruit in the image. If there are multiple fruits on the picture use plural form. Answer with one word: Apple (Apples) or Pomegranate (Pomegranates)?"
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_url
                            }
                        }
                    ]
                }
            ],
        )
        
        answer = completion.choices[0].message.content.strip()
        return answer
    except Exception as e:
        print(f"Ошибка при запросе к модели {model_id} для изображения {image_url}: {e}")
        return "Error"

def normalize_prediction(pred):
    """Нормализация предсказания"""
    pred_lower = pred.lower()
    if 'apple' in pred_lower:
        return 'Apple'
    elif 'pomegranate' in pred_lower:
        return 'Pomegranate'
    else:
        return 'Unknown'

In [44]:
def calculate_accuracy(true_labels, predictions):
    """Вычисление точности (Accuracy)"""
    correct = 0
    for true, pred in zip(true_labels, predictions):
        if true == pred:
            correct += 1
    return correct / len(true_labels) if true_labels else 0

In [45]:
def calculate_precision_recall(true_labels, predictions, positive_class='Apple'):
    """Вычисление точности (Precision) и полноты (Recall)"""
    tp = 0  # True Positive
    fp = 0  # False Positive
    fn = 0  # False Negative
    
    for true, pred in zip(true_labels, predictions):
        if pred == positive_class:
            if true == positive_class:
                tp += 1
            else:
                fp += 1
        elif true == positive_class and pred != positive_class:
            fn += 1
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    return precision, recall, tp, fp, fn

In [46]:
def calculate_f1_score(precision, recall):
    """Вычисление F1-меры"""
    if precision + recall == 0:
        return 0
    return 2 * (precision * recall) / (precision + recall)

In [47]:
def calculate_confusion_matrix(true_labels, predictions, classes=['Apple', 'Pomegranate']):
    """Вычисление матрицы ошибок"""
    cm = {class1: {class2: 0 for class2 in classes} for class1 in classes}
    
    for true, pred in zip(true_labels, predictions):
        cm[true][pred] += 1
    
    return cm

In [48]:
def calculate_specificity(true_labels, predictions, negative_class='Pomegranate'):
    """Вычисление специфичности (True Negative Rate)"""
    tn = 0  # True Negative
    fp = 0  # False Positive
    
    for true, pred in zip(true_labels, predictions):
        if true == negative_class:
            if pred == negative_class:
                tn += 1
            else:
                fp += 1
    
    return tn / (tn + fp) if (tn + fp) > 0 else 0

In [49]:
def test_model(model_id, model_name, image_urls, true_labels):
    """Тестирование одной модели"""
    print(f"Тестирование модели: {model_name}")
    predictions = []
    
    for i, image_url in enumerate(image_urls):
        print(f"  Изображение {i+1}/{len(image_urls)}")
        pred = get_prediction(model_id, image_url)
        normalized_pred = normalize_prediction(pred)
        predictions.append(normalized_pred)
        print(f"    Предсказание: {pred} -> {normalized_pred}")
        
        # Задержка чтобы не превысить лимиты API
        time.sleep(20)
    
    # Вычисление метрик
    accuracy = calculate_accuracy(true_labels, predictions)
    precision, recall, tp, fp, fn = calculate_precision_recall(true_labels, predictions)
    f1 = calculate_f1_score(precision, recall)
    specificity = calculate_specificity(true_labels, predictions)
    confusion_matrix = calculate_confusion_matrix(true_labels, predictions)
    
    metrics = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Specificity': specificity,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'Confusion_Matrix': confusion_matrix,
        'Predictions': predictions
    }
    
    return metrics

In [50]:
def print_metrics(metrics, model_name):
    """Вывод метрик для модели"""
    print(f"\n--- Результаты для {model_name} ---")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall: {metrics['Recall']:.4f}")
    print(f"F1-Score: {metrics['F1-Score']:.4f}")
    print(f"Specificity: {metrics['Specificity']:.4f}")
    print(f"TP: {metrics['TP']}, FP: {metrics['FP']}, FN: {metrics['FN']}")
    print("Confusion Matrix:")
    for true_class in metrics['Confusion_Matrix']:
        for pred_class in metrics['Confusion_Matrix'][true_class]:
            count = metrics['Confusion_Matrix'][true_class][pred_class]
            print(f"  {true_class} -> {pred_class}: {count}")
    print("-" * 50)

In [51]:
results = {}

for model_name, model_id in models.items():
    metrics = test_model(model_id, model_name, image_urls, true_labels)
    results[model_name] = metrics
    print_metrics(metrics, model_name)

Тестирование модели: Qwen
  Изображение 1/18
    Предсказание: Apple (Apples) -> Apple
  Изображение 2/18
    Предсказание: Apples -> Apple
  Изображение 3/18
    Предсказание: Apple (Apples) -> Apple
  Изображение 4/18
    Предсказание: Apple (Apples) -> Apple
  Изображение 5/18
    Предсказание: Apples -> Apple
  Изображение 6/18
    Предсказание: Apples -> Apple
  Изображение 7/18
    Предсказание: Apple (Apples) -> Apple
  Изображение 8/18
    Предсказание: Apple -> Apple
  Изображение 9/18
    Предсказание: Apples -> Apple
  Изображение 10/18
    Предсказание: Pomegranate -> Pomegranate
  Изображение 11/18
    Предсказание: Pomegranate (Pomegranates) -> Pomegranate
  Изображение 12/18
    Предсказание: Pomegranate (Pomegranates) -> Pomegranate
  Изображение 13/18
    Предсказание: Pomegranate (Pomegranates) -> Pomegranate
  Изображение 14/18
    Предсказание: Pomegranate (Pomegranates) -> Pomegranate
  Изображение 15/18
    Предсказание: Pomegranate -> Pomegranate
  Изображение 16

In [52]:
# Сравнительный анализ
print("\n=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ===")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']:
    model1_metric = results[list(models.keys())[0]][metric]
    model2_metric = results[list(models.keys())[1]][metric]
    
    print(f"{metric}:")
    print(f"  {list(models.keys())[0]}: {model1_metric:.4f}")
    print(f"  {list(models.keys())[1]}: {model2_metric:.4f}")
    print(f"  Разница: {abs(model1_metric - model2_metric):.4f}")
    
    if model1_metric > model2_metric:
        print(f"  Лучше: {list(models.keys())[0]}")
    elif model2_metric > model1_metric:
        print(f"  Лучше: {list(models.keys())[1]}")
    else:
        print(f"  Равны")
    print()


=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ===
Accuracy:
  Qwen: 1.0000
  Gemma: 1.0000
  Разница: 0.0000
  Равны

Precision:
  Qwen: 1.0000
  Gemma: 1.0000
  Разница: 0.0000
  Равны

Recall:
  Qwen: 1.0000
  Gemma: 1.0000
  Разница: 0.0000
  Равны

F1-Score:
  Qwen: 1.0000
  Gemma: 1.0000
  Разница: 0.0000
  Равны

Specificity:
  Qwen: 1.0000
  Gemma: 1.0000
  Разница: 0.0000
  Равны



In [53]:
# Вывод предсказаний для детального анализа
print("\n=== ДЕТАЛЬНЫЕ ПРЕДСКАЗАНИЯ ===")
for i, image_url in enumerate(image_urls):
    print(f"Изображение {i+1}: {image_url}")
    print(f"  Истинная метка: {true_labels[i]}")
    for model_name in models:
        pred = results[model_name]['Predictions'][i]
        print(f"  {model_name}: {pred}")
    print()


=== ДЕТАЛЬНЫЕ ПРЕДСКАЗАНИЯ ===
Изображение 1: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple1.jpeg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 2: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple2.jpeg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 3: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple3.jpg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 4: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple4.jpg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 5: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple5.jpg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 6: https://raw.githubusercontent.com/Vlad230103/hw1/refs/heads/master/HW3/apple6.jpg
  Истинная метка: Apple
  Qwen: Apple
  Gemma: Apple

Изображение 7: https://raw.githubusercontent.com/Vlad230103/